# Lab 9: A/B Testing — Diagnosing Experimental Failures
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 112 min Core + 10 min Extension

---

**Format:** This lab contains **deliberately flawed analyses**. Your job:
1. Run the code
2. Identify what is wrong (you are not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

**Time:** Core (112 min) + Extension (10 min) + Challenge (optional, ungraded)

With setup, the README and the GitHub submission, plan on about two and a quarter hours, most of it after class.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 1:** fix the analyst's CUPED code. A copy of it, with the same three bugs, sits in an editable cell; you change about three lines. A check cell after it tests each fix.
> - **Part 3, Step B:** two blanks: the bias against the global rollout, and its share of the naive ATE.
> - **Part 4, Step B:** four blanks: the posterior parameters for each variant.
> - **Write it yourself (required, after Part 4):** a `bayes_ab()` function and two lines that use it, about eight lines, in an empty cell. A check cell tells you when it is right. It is required.
> - **Challenge (optional, ungraded):** CUPED on the LaLonde data, from a four-step outline.
>
> **You write, in text cells:** one sentence per bug in Part 1, two answers in Part 2, and in Part 3 two short answers (below Steps B and C) and one paragraph.
>
> **You run and read:** everything else. That is the simulated A/B test and the analyst's CUPED report, the two SRM experiments, the three marketplace worlds with Part 3's Steps A, C and D, the conversion data and Part 4's Steps C to G, the `cuped_utils` module, and the whole Extension. Read each cell before you run it: the questions ask about what it prints, and Write it yourself rebuilds Part 4's Steps C and D as a function.

### Learning Objectives

By the end of this lab, you will be able to:

1. **Implement and debug CUPED** (Controlled-experiment Using Pre-Experiment Data) to reduce variance in A/B test estimates
2. **Detect Sample Ratio Mismatch (SRM)** using chi-squared tests and reason about when statistical significance reflects a real problem vs. noise
3. **Identify SUTVA violations** in two-sided marketplace experiments where interference between units biases naive treatment effect estimates
4. **Compare Bayesian and frequentist A/B testing** frameworks, including posterior updating, expected loss, and permutation tests

### Data

- **Simulated data** in every Part: an A/B test with a pre-period covariate (Part 1), two experiments' user counts (Part 2), a two-sided marketplace (Part 3), a conversion experiment (Part 4) and a peeking study (Extension)
- **LaLonde NSW** (National Supported Work), used in the take-home Challenge: a classic job training RCT with pre-experiment earnings (`re74`, `re75`) and post-experiment earnings (`re78`)

### Packages

`numpy`, `scipy`, `pandas`, `matplotlib`

In [ ]:
# GIVEN — run as-is
# Step 0: import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
# "beta as beta_dist" imports scipy's Beta distribution under a longer name, so a
# variable called beta cannot overwrite it
from scipy.stats import chi2, beta as beta_dist, permutation_test

RANDOM_STATE = 42

---

## Part 1: Diagnose Broken CUPED (35 min)

**CUPED** (Controlled-experiment Using Pre-Experiment Data) is a variance reduction technique used at Microsoft, Netflix, and Booking.com. The idea: if you have a pre-experiment covariate $X_{\text{pre}}$ correlated with the outcome $Y$, you can construct an adjusted outcome:

$$Y_{\text{adj}} = Y - \theta \cdot (X_{\text{pre}} - \bar{X}_{\text{pre}})$$

where $\theta = \text{Cov}(Y, X_{\text{pre}}) / \text{Var}(X_{\text{pre}})$.

The adjusted outcome has lower variance than the raw outcome, so your standard errors shrink — meaning you can detect smaller effects or reach significance faster.

The standard error of a difference between two group means is
$\sqrt{s_1^2 / n_1 + s_2^2 / n_2}$, where $s_1^2$ and $s_2^2$ are the two groups' sample variances.

A data scientist at a tech company implemented CUPED for their A/B test. **Their code contains THREE bugs.** Run it, identify each bug, and fix it.

**Reading the code.** `y[treatment == 1]` keeps the entries of `y` where `treatment`
is 1: the comparison gives one True/False per user, and indexing with it keeps the Trues.
`np.var(..., ddof=1)` is the sample variance, which divides by $n - 1$. In the prints,
`{x:.4f}` shows four decimals and `{x:.1f}` one.

In [ ]:
# GIVEN — run as-is
# Step 1: simulate an A/B test with pre-experiment data

np.random.seed(RANDOM_STATE)

n_per_group = 5000
true_ate = 2.0          # the true treatment effect: +$2 of revenue per user

# Pre-experiment revenue, one draw per user
x_pre_control = np.random.normal(50, 10, size=n_per_group)
x_pre_treat = np.random.normal(50, 10, size=n_per_group)

# Post-experiment revenue: 0.7 x pre-period revenue, plus noise, plus the effect for the treated
noise_control = np.random.normal(0, 3.2, size=n_per_group)
noise_treat = np.random.normal(0, 3.2, size=n_per_group)
y_control = 0.7 * x_pre_control + noise_control
y_treat = 0.7 * x_pre_treat + noise_treat + true_ate

# One array each for all 10,000 users: the control group first, then the treated
y = np.concatenate([y_control, y_treat])
x_pre = np.concatenate([x_pre_control, x_pre_treat])
treatment = np.concatenate([np.zeros(n_per_group), np.ones(n_per_group)])    # 0 = control, 1 = treated

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 1, continued: the data scientist's CUPED estimate, and the naive one for comparison

# np.corrcoef returns a 2 x 2 matrix; [0, 1] picks the entry off the diagonal
theta = np.corrcoef(y, x_pre)[0, 1]
print(f"Theta (data scientist's): {theta:.4f}")

y_adj = y - theta * x_pre

y_adj_treat = y_adj[treatment == 1]
y_adj_control = y_adj[treatment == 0]

se_cuped = np.sqrt(
    np.var(y[treatment == 1], ddof=1) / n_per_group
    + np.var(y[treatment == 0], ddof=1) / n_per_group
)

ate_cuped = np.mean(y_adj_treat) - np.mean(y_adj_control)

# The naive estimate: the plain difference in means, with no adjustment
ate_naive = np.mean(y_treat) - np.mean(y_control)
se_naive = np.sqrt(
    np.var(y_treat, ddof=1) / n_per_group
    + np.var(y_control, ddof=1) / n_per_group
)

print(f"\n=== Data Scientist's CUPED Report ===")
print(f"Naive ATE:       {ate_naive:.4f}  (SE = {se_naive:.4f})")
print(f"CUPED ATE:       {ate_cuped:.4f}  (SE = {se_cuped:.4f})")
print(f"Variance reduction: {(1 - (se_cuped / se_naive)**2) * 100:.1f}%")

### YOUR DIAGNOSIS

The report has two symptoms. CUPED should cut the variance by roughly the share of
the outcome's variance that the pre-period covariate explains, and the reported
reduction is exactly 0.0%. The ATE also moved, from 1.9107 to 2.0502. CUPED may move
it a little, because it corrects for the chance difference in $X_{\text{pre}}$ between
the two arms.

The code has **three bugs**. For each:

1. **Identify** the line with the bug
2. **Explain** why it is wrong: which formula at the top of Part 1 does it break?
3. **Fix** it in the editable copy below

**Hints:** compare each line of the analyst's CUPED code with the formulas at the top
of Part 1, one symbol at a time. Then, for each number in the report, ask which
variable it was computed from.

**Verification checkpoint:** after all three fixes, $\theta$ is **0.6965**, the mean
of `y_adj` is **36.0284**, the variance reduction is **82.7%** (the SE falls from 0.1539 to 0.0641), and the CUPED
ATE is **2.0185** against a true ATE of 2.0. That is CUPED's ceiling on this data: it
cannot remove more variance than the pre-period covariate explains, which here is
$0.7^2 \cdot 100 / (0.7^2 \cdot 100 + 3.2^2) = 82.7\%$. One of the three bugs changes
neither the variance reduction nor the ATE, so 82.7% alone does not prove you are
done: the check cell tests each bug on its own line.

In [ ]:
# YOUR TASK — fix the analyst's CUPED code
# This is the analyst's code from Step 1 again, with the same three bugs. Correct them
# here, then run the check below. Leave the analyst's cell above as it is.

theta = np.corrcoef(y, x_pre)[0, 1]
print(f"Theta: {theta:.4f}")

y_adj = y - theta * x_pre

y_adj_treat = y_adj[treatment == 1]
y_adj_control = y_adj[treatment == 0]

se_cuped = np.sqrt(
    np.var(y[treatment == 1], ddof=1) / n_per_group
    + np.var(y[treatment == 0], ddof=1) / n_per_group
)

ate_cuped = np.mean(y_adj_treat) - np.mean(y_adj_control)
var_reduction = (1 - (se_cuped / se_naive)**2) * 100

print(f"Naive ATE:       {ate_naive:.4f}  (SE = {se_naive:.4f})")
print(f"CUPED ATE:       {ate_cuped:.4f}  (SE = {se_cuped:.4f})")
print(f"True ATE:        {true_ate:.4f}")
print(f"Variance reduction: {var_reduction:.1f}%")

In [ ]:
# CHECK — run as-is, after your fix
# Each entry pairs a description with True or False; every one should print PASS

checks = {
    "theta is 0.6965": abs(theta - 0.6965) < 0.0005,
    "mean of y_adj is 36.0284": abs(np.mean(y_adj) - 36.0284) < 0.0001,
    "CUPED SE is 0.0641": abs(se_cuped - 0.0641) < 0.00005,
    "variance reduction is 82.7%": abs(var_reduction - 82.7) < 0.1,
    "CUPED ATE is 2.0185": abs(ate_cuped - 2.0185) < 0.0005,
}

for name, passed in checks.items():
    # "a if condition else b" is a if the condition holds, otherwise b
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {name}")

# all() is True only when every value in the dict is True
if all(checks.values()):
    print("\nAll Part 1 checks passed.")

### Document Your Diagnosis

For each bug, write one sentence explaining: (a) what was wrong, and (b) why it matters.

*Bug 1:*

*Bug 2:*

*Bug 3:*

---

## Part 2: Sample Ratio Mismatch Detection (12 min)

**Sample Ratio Mismatch (SRM)** occurs when the observed split between treatment and control deviates from the intended split. SRM is a red flag — it usually means something in the assignment mechanism is broken (e.g., a bot filtering bug, a redirect loop, a misconfigured load balancer).

Detecting SRM is straightforward: a chi-squared goodness-of-fit test against the expected proportions. Industry practice uses a strict threshold — **p < 0.001**, not 0.05 — precisely because the test is run on every experiment and a 5% false-alarm rate would drown the alerting channel. The critical value at that threshold with df = 1 is **10.83**.

The harder question is: **when does a statistically significant SRM actually matter?**

Below are two experiments. Both trip the p < 0.001 alarm. Your job is to decide which one represents a real problem.

**Reading the prints.** `{n:,}` prints a whole number with thousands separators.
`{p:.3g}` prints three significant digits, and switches to scientific notation
(`7.74e-06` means $7.74 \times 10^{-6}$) for very small numbers.

In [ ]:
# GIVEN — run as-is
# Step 2: the SRM test

SRM_ALARM = 0.001          # industry threshold; chi-squared critical value 10.83


def srm_test(n_treat, n_control, expected_ratio=0.5):
    """Chi-squared goodness-of-fit test of the treatment/control split."""
    n_total = n_treat + n_control
    observed = np.array([n_treat, n_control])
    expected = np.array([expected_ratio, 1 - expected_ratio]) * n_total

    # Chi-squared statistic: sum of (O - E)^2 / E
    chi2_stat = np.sum((observed - expected) ** 2 / expected)
    # chi2.cdf(x) is P(statistic <= x), so 1 minus it is the upper tail: the p-value
    p_value = 1 - chi2.cdf(chi2_stat, df=1)

    return chi2_stat, p_value      # gives back both values at once, as a pair (a tuple)

In [ ]:
# GIVEN — run as-is
# Step 2, continued: two experiments with SRM


def srm_report(name, n_treat, n_control):
    """Print one experiment's split and its SRM test."""
    n_total = n_treat + n_control
    chi2_stat, p_value = srm_test(n_treat, n_control)     # unpack the pair into two names
    print(f"=== {name} ===")
    print(f"Treatment: {n_treat:,}  ({n_treat / n_total * 100:.2f}%)")
    print(f"Control:   {n_control:,}  ({n_control / n_total * 100:.2f}%)")
    print(f"Deviation from 50/50: {abs(n_treat / n_total - 0.5) * 100:.3f} pp")
    print(f"Chi-squared: {chi2_stat:.2f}")
    print(f"P-value: {p_value:.3g}")
    print(f"SRM alarm (p < {SRM_ALARM}): {p_value < SRM_ALARM}")


# 50_600 is 50600: Python ignores the underscore, which only makes the number easier to read
srm_report("Experiment A: 100K Users", 50_600, 49_400)       # 50.6% / 49.4%
print()
srm_report("Experiment B: 20M Users", 10_010_000, 9_990_000)  # 50.05% / 49.95%

### YOUR TASK

Both alarms fire, yet Experiment B's imbalance is twelve times smaller than
Experiment A's and its chi-squared statistic is larger.

Answer these questions in the markdown cell below:

1. **Both experiments trip the p < 0.001 alarm.** Experiment A sits 0.6 percentage points off the intended 50%, Experiment B 0.05 points. Which is more likely to indicate a real engineering problem, and which imbalance is more concerning for the validity of the experiment's results? Why?
2. **The paradox:** With enough users, the test alarms on a real misassignment far too small to bias anything. How should an experimentation platform handle this? (Hint: think about effect sizes and about what magnitude of misassignment could actually move the metric — not just p-values.)

**Verification checkpoint:** The measured statistics are

| | N | split | deviation | chi-squared | p |
|---|---:|---|---:|---:|---:|
| A | 100,000 | 50.60 / 49.40 | 0.60 pp | **14.40** | 0.000148 |
| B | 20,000,000 | 50.05 / 49.95 | 0.05 pp | **20.00** | 7.7e-06 |

Both exceed the 10.83 critical value for p < 0.001 at df = 1. Note that B's
chi-squared is the *larger* of the two despite an imbalance twelve times
smaller: the statistic scales with $N$ times the *squared fractional*
deviation, so N does the work.

*Your answers here:*

1.

2.

---

## Part 3: SUTVA Violation Simulation (20 min)

**SUTVA** (Stable Unit Treatment Value Assumption) requires that one unit's treatment assignment does not affect another unit's outcome. In a **two-sided marketplace** it can fail: buyers given a discount (the treatment) buy more, and that leaves less inventory for everyone else.

The simulation below has 10,000 buyers and 4,000 items. Buyers arrive in random order. A buyer's purchase probability is 0.30, plus 0.15 with the discount, times a factor for how much inventory is left. `availability` is the share of items still on the shelf, and the factor is `availability ** 1.0` for a control buyer and `availability ** 0.3` for a discounted one: a discounted buyer converts on the spot, while an undiscounted one hesitates and may come back to find the item gone.

Holtz et al. (2025) estimate the bias from interference in a large marketplace experiment at roughly 20% or more of the naive estimate.

Below you will simulate three worlds — no interference, individual randomization with interference, and a full rollout — and measure the bias against two benchmarks.

In [ ]:
# GIVEN — run as-is
# Step 3: simulate a two-sided marketplace. World 1: no interference

np.random.seed(RANDOM_STATE)

n_buyers = 10_000
n_items = 4_000          # finite inventory: far fewer items than buyers

# Randomly assign 50% of buyers to treatment (a price discount): 1 = treated, 0 = control
treatment_buyer = np.random.binomial(1, 0.5, size=n_buyers)

base_prob = 0.30         # every buyer's base purchase probability
treat_boost = 0.15       # the discount adds 15 percentage points

# Availability elasticity: how hard a picked-over inventory hits you.
# Control buyers hesitate and lose the item (1.0);
# discounted buyers convert on the spot and mostly still get it (0.3).
ELAST_CONTROL = 1.0
ELAST_TREAT = 0.3

# World 1: each buyer's purchase is independent of every other buyer's (SUTVA holds).
# treatment_buyer is 0 or 1, so this is 0.30 for control buyers and 0.45 for treated ones.
prob_no_interference = base_prob + treat_boost * treatment_buyer
purchase_no_interference = np.random.binomial(1, prob_no_interference)

ate_no_interference = (
    np.mean(purchase_no_interference[treatment_buyer == 1])
    - np.mean(purchase_no_interference[treatment_buyer == 0])
)

print("=== World 1: No Interference (SUTVA Holds) ===")
print(f"Treated purchase rate:  {np.mean(purchase_no_interference[treatment_buyer == 1]):.4f}")
print(f"Control purchase rate:  {np.mean(purchase_no_interference[treatment_buyer == 0]):.4f}")
print(f"Naive ATE:              {ate_no_interference:.4f}")
print(f"True individual effect: {treat_boost:.4f}")

In [ ]:
# GIVEN — run as-is
# Step 3, continued: World 2, individual randomization with interference


def run_marketplace(assignment, seed=42):
    """One pass of the marketplace. Buyers arrive in random order and compete
    for a fixed inventory; treated buyers are less sensitive to scarcity."""
    np.random.seed(seed)
    arrival_order = np.random.permutation(n_buyers)      # the buyer numbers, shuffled
    remaining_items = n_items
    purchases = np.zeros(n_buyers, dtype=int)

    for buyer_idx in arrival_order:
        availability = remaining_items / n_items         # 1 when the shelves are full, 0 when empty
        # a if condition else b: the treated elasticity for a treated buyer, else the control one
        elasticity = ELAST_TREAT if assignment[buyer_idx] == 1 else ELAST_CONTROL
        prob_i = (base_prob + treat_boost * assignment[buyer_idx]) * availability ** elasticity

        purchased = np.random.binomial(1, prob_i)
        purchases[buyer_idx] = purchased
        if purchased and remaining_items > 0:
            remaining_items -= 1

    return purchases, remaining_items


# run_marketplace returns two values; unpack them into two names
purchases_interference, remaining_items = run_marketplace(treatment_buyer)

ate_interference = (
    np.mean(purchases_interference[treatment_buyer == 1])
    - np.mean(purchases_interference[treatment_buyer == 0])
)

print("=== World 2: Individual Randomization, With Interference ===")
print(f"Treated purchase rate:  {np.mean(purchases_interference[treatment_buyer == 1]):.4f}")
print(f"Control purchase rate:  {np.mean(purchases_interference[treatment_buyer == 0]):.4f}")
print(f"Naive ATE:              {ate_interference:.4f}")
print(f"Items remaining:        {remaining_items:,} of {n_items:,}")

In [ ]:
# GIVEN — run as-is
# Step 3, continued: World 3, a global rollout against no rollout.
# Ship to EVERYONE vs ship to NO ONE, with the same inventory both times.

purchases_all_treated, rem_all = run_marketplace(np.ones(n_buyers, dtype=int))
purchases_none_treated, rem_none = run_marketplace(np.zeros(n_buyers, dtype=int))
ate_global = np.mean(purchases_all_treated) - np.mean(purchases_none_treated)

print("=== World 3: Global Rollout vs No Rollout ===")
print(f"Purchase rate, everyone treated: {np.mean(purchases_all_treated):.4f}  "
      f"(items left {rem_all:,})")
print(f"Purchase rate, nobody treated:   {np.mean(purchases_none_treated):.4f}  "
      f"(items left {rem_none:,})")
print(f"Global ATE (everyone treated minus nobody treated): {ate_global:.4f}")

### YOUR TASK

1. **Step A (given).** The bias against the true individual effect (0.15), as a percentage of the naive ATE in World 2.
2. **Step B (yours).** The bias against the *global* ATE in World 3, the contrast between shipping to everyone and shipping to nobody, as a percentage of the naive ATE too. The question below Step B asks which benchmark matters.
3. **Step C (given).** How far each arm's purchase rate fell from World 1 to World 2. The question below Step C asks what that does to the measured gap.
4. **Step D (given).** A chart of the purchase rates for the four groups.

**Verification checkpoint:** The measured numbers are

| | treated rate | control rate | ATE |
|---|---:|---:|---:|
| World 1 (no interference) | 0.4547 | 0.3050 | 0.1497 |
| World 2 (interference)    | 0.3729 | 0.1868 | **0.1861** |
| World 3 (global rollout)  | 0.3545 | 0.2117 | 0.1428 |

So the naive ATE overstates the individual effect by 0.0361, or **19.4% of the
naive estimate**, and overstates the global rollout effect by 0.0433, or
**23.3%**. The control rate falls by 0.118 between Worlds 1 and 2; the treated
rate falls by 0.082.

In [ ]:
# GIVEN — run as-is
# Step A: the bias against the true individual effect, and its share of the naive ATE

true_individual_effect = treat_boost
interference_bias = ate_interference - true_individual_effect
bias_pct = interference_bias / ate_interference * 100

print(f"True individual effect:  {true_individual_effect:.4f}")
print(f"Naive ATE (World 2):     {ate_interference:.4f}")
print(f"Interference bias:       {interference_bias:.4f}")
print(f"Bias as % of naive ATE:  {bias_pct:.1f}%")

In [ ]:
# YOUR TASK — fill in the blanks, as Step A does for the individual effect
# Step B: the bias against the global rollout, the benchmark the decision needs

global_bias = ____
global_bias_pct = ____

print(f"Naive ATE (World 2):    {ate_interference:.4f}")
print(f"Global ATE (World 3):   {ate_global:.4f}")
print(f"Overstatement:          {global_bias:.4f}")
print(f"As % of naive ATE:      {global_bias_pct:.1f}%")

**Step B:** which of the two benchmarks should a product manager care about, and why?

*Your answer here:*

In [ ]:
# GIVEN — run as-is
# Step C: trace the mechanism. How far did each arm's purchase rate fall from World 1 to World 2?

drop_treat = (np.mean(purchase_no_interference[treatment_buyer == 1])
              - np.mean(purchases_interference[treatment_buyer == 1]))
drop_control = (np.mean(purchase_no_interference[treatment_buyer == 0])
                - np.mean(purchases_interference[treatment_buyer == 0]))

print(f"Treated rate fell by: {drop_treat:.4f}")
print(f"Control rate fell by: {drop_control:.4f}")

**Step C:** which arm's purchase rate fell further, and why does that widen the
measured gap?

*Your answer here:*

In [ ]:
# GIVEN — run as-is
# Step D: the four purchase rates, with and without interference

rates = {
    "Treat\n(No Interference)": np.mean(purchase_no_interference[treatment_buyer == 1]),
    "Control\n(No Interference)": np.mean(purchase_no_interference[treatment_buyer == 0]),
    "Treat\n(Interference)": np.mean(purchases_interference[treatment_buyer == 1]),
    "Control\n(Interference)": np.mean(purchases_interference[treatment_buyer == 0]),
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(rates.keys(), rates.values(), color=["steelblue", "steelblue", "sienna", "sienna"])
ax.bar_label(bars, fmt="%.3f")          # write each bar's height above it
ax.set_ylabel("Purchase Rate")
ax.set_title("Purchase Rates: Interference Depresses Control Outcomes")
ax.set_ylim(0, 0.55)
plt.tight_layout()
plt.show()

### Document Your Analysis

Write one paragraph: why does interference make the naive ATE *overstate* the
effect in this marketplace? Would it still overstate if scarcity multiplied both
arms' purchase probability by the same availability factor? Say why or why not.


*Your explanation here:*

---

## Part 4: Bayesian A/B Testing (15 min)

The frequentist framework answers: "If there is no effect, how often would we see data this extreme?"

The Bayesian framework answers: "Given the data, what is the probability that B is better than A?"

For conversion rate experiments, the **Beta-Binomial conjugate model** provides closed-form posterior updates:
- Prior: $\theta \sim \text{Beta}(\alpha, \beta)$
- Likelihood: $X \sim \text{Binomial}(n, \theta)$
- Posterior: $\theta | X \sim \text{Beta}(\alpha + X, \beta + n - X)$

You will implement this model and compare its conclusions to a frequentist permutation test.

In [ ]:
# GIVEN — run as-is
# Step 4: simulate a conversion rate experiment

np.random.seed(RANDOM_STATE)

# Variant A (control) converts 12% of users; variant B (treatment) 13.5%: a true lift of 1.5 pp
n_a = 3000
n_b = 3000
true_rate_a = 0.120
true_rate_b = 0.135

# One 1 (converted) or 0 (did not) per user
conversions_a = np.random.binomial(1, true_rate_a, size=n_a)
conversions_b = np.random.binomial(1, true_rate_b, size=n_b)

successes_a = conversions_a.sum()
successes_b = conversions_b.sum()
failures_a = n_a - successes_a
failures_b = n_b - successes_b

print("=== Observed Experiment Data ===")
print(f"Variant A: {successes_a} conversions / {n_a} users = {successes_a/n_a:.4f}")
print(f"Variant B: {successes_b} conversions / {n_b} users = {successes_b/n_b:.4f}")
print(f"Observed lift: {(successes_b/n_b - successes_a/n_a)*100:.2f} pp")

### YOUR TASK

You write Step B; Steps A and C to G are given. Read Steps C and D closely: Write it
yourself asks you to rebuild them as a function.

1. **Set priors** (Step A, given): Beta(1, 1), the uniform prior (no prior information)
2. **Update posteriors** (Step B, yours): the posterior Beta parameters for both variants
3. **P(B > A)** (Step C, given): from 100,000 draws of each posterior, the probability that variant B's true rate exceeds variant A's
4. **Expected loss** (Step D, given): the expected loss of shipping B, $E[\max(\theta_A - \theta_B, 0)]$, the conversion rate you give up in the draws where A is better; and the expected loss of shipping A, the same with the roles swapped
5. **Decision** (Step E, given): ship B if P(B > A) > 0.95 AND the expected loss of shipping B is < 0.001 (0.1 pp)
6. **Frequentist comparison** (Step F, given): a permutation test, to compare the conclusion

**Reading the code.** `beta_dist.rvs(a, b, size=n, random_state=s)` draws `n` values
from Beta(a, b); `random_state` seeds the draw, so it gives the same numbers every
run. `samples_b > samples_a` compares the two arrays draw by draw and gives one
True/False per draw, and the mean of True/False values is the share of Trues.
`np.maximum(x, 0)` replaces each negative entry of `x` with 0. `a and b` is True only
when both are True.

The permutation test works as in Lab 8: `permutation_test` shuffles the A and B labels
9,999 times and hands the statistic function many shuffles at once, so the function
gets an `axis` argument that it must pass on to every mean it takes.

In [ ]:
# YOUR TASK — fill in the blanks
# Steps A and B: Beta(1, 1) priors, and the posterior for each variant

prior_alpha = 1
prior_beta = 1

post_alpha_a = ____
post_beta_a = ____
post_alpha_b = ____
post_beta_b = ____

print(f"Posterior A: Beta({post_alpha_a}, {post_beta_a})")
print(f"Posterior B: Beta({post_alpha_b}, {post_beta_b})")
print(f"Posterior mean A: {post_alpha_a / (post_alpha_a + post_beta_a):.4f}")
print(f"Posterior mean B: {post_alpha_b / (post_alpha_b + post_beta_b):.4f}")

In [ ]:
# GIVEN — run as-is
# Step C: P(B > A) from 100,000 draws of each posterior

n_mc = 100_000
samples_a = beta_dist.rvs(post_alpha_a, post_beta_a, size=n_mc, random_state=RANDOM_STATE)
samples_b = beta_dist.rvs(post_alpha_b, post_beta_b, size=n_mc, random_state=RANDOM_STATE + 1)

prob_b_better = np.mean(samples_b > samples_a)      # the share of draws in which B is higher
print(f"P(B > A) = {prob_b_better:.4f}")

In [ ]:
# GIVEN — run as-is
# Step D: the expected loss of shipping each variant

# Shipping B loses theta_A - theta_B in a draw where A is better, and nothing otherwise
expected_loss_b = np.mean(np.maximum(samples_a - samples_b, 0))
expected_loss_a = np.mean(np.maximum(samples_b - samples_a, 0))
print(f"Expected loss of shipping B: {expected_loss_b:.6f}")
print(f"Expected loss of shipping A: {expected_loss_a:.6f}")

In [ ]:
# GIVEN — run as-is
# Step E: the ship rule

ship_threshold_prob = 0.95
ship_threshold_loss = 0.001     # 0.1 percentage points
should_ship = prob_b_better > ship_threshold_prob and expected_loss_b < ship_threshold_loss

print(f"P(B > A) = {prob_b_better:.4f} (threshold: {ship_threshold_prob})")
print(f"Expected loss = {expected_loss_b:.6f} (threshold: {ship_threshold_loss})")
print(f"Ship B: {should_ship}")

In [ ]:
# GIVEN — run as-is
# Step F: a permutation test on the same data


def stat_func(x, y, axis):
    """Difference in conversion rates, x minus y."""
    return np.mean(x, axis=axis) - np.mean(y, axis=axis)


result = permutation_test(
    (conversions_b, conversions_a),
    stat_func,
    n_resamples=9999,
    alternative="two-sided",
    random_state=RANDOM_STATE,
)
print("=== Frequentist Permutation Test ===")
print(f"Observed difference: {result.statistic:.4f}")
print(f"P-value: {result.pvalue:.4f}")
print(f"Decision at alpha=0.05: {'Reject H0' if result.pvalue < 0.05 else 'Fail to reject H0'}")

In [ ]:
# GIVEN — run as-is
# Step G: the two posterior distributions

x_grid = np.linspace(0.08, 0.18, 1000)          # 1,000 conversion rates from 8% to 18%

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_grid, beta_dist.pdf(x_grid, post_alpha_a, post_beta_a), color="steelblue", label="Posterior A")
ax.plot(x_grid, beta_dist.pdf(x_grid, post_alpha_b, post_beta_b), color="sienna", label="Posterior B")
ax.set_xlabel("Conversion Rate")
ax.set_ylabel("Posterior Density")
ax.set_title(f"Bayesian A/B Test: P(B > A) = {prob_b_better:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

**Verification checkpoint.** The measured values are

| | A | B |
|---|---|---|
| posterior | Beta(358, 2644) | Beta(389, 2613) |
| posterior mean | 0.1193 | 0.1296 |
| expected loss of shipping | 0.010770 | **0.000473** |

$P(B > A) = $ **0.8866**, so the ship rule says **False**: the loss criterion passes, the
probability criterion does not. The permutation test gives an observed difference of
0.0103 and **p = 0.2302**: fail to reject at 5%.

---

## Write it yourself (required — 25 min)

This is the one function in this lab you write from an empty cell, and it is
required. No AI: write it yourself.

Part 4's test did not ship B: with 3,000 users per arm, P(B > A) was 0.8866, short of
0.95. Would a similar lift ship with more data?

In the empty cell below:
1. Write a function `bayes_ab(successes_a, n_a, successes_b, n_b, n_mc=100_000, seed=42)`
   that returns two numbers, in this order: P(B > A), and the expected loss of
   shipping B. Use Beta(1, 1) priors, as Part 4 did, and draw A's samples with
   `random_state=seed` and B's with `random_state=seed + 1`, as Step C did with 42 and 43.
2. Use it on a run of the test four times as long: 12,000 users per arm, with 1,428
   conversions for A and 1,560 for B. Set `longer_prob, longer_loss` to its two
   results, and `longer_ship` to the decision under Part 4's rule (Step E).

It takes about eight lines, and Part 4's Steps B to D are the model. A function can return two
values as `return x, y`, and `p, loss = bayes_ab(...)` unpacks them. Do Part 4 first:
the check runs your function on Part 4's counts, where it should reproduce Part 4's
checkpoint, then on a few small inputs, where each common slip gives a different
answer. Until you write the cell, the check says "Not written yet"; once you do, it
fails with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: bayes_ab, then longer_prob, longer_loss and longer_ship

In [ ]:
# CHECK — for bayes_ab above
# globals() holds every name defined so far in this notebook
if "bayes_ab" not in globals():
    print("Not written yet: define bayes_ab in the cell above, then run this check again.")
else:
    import inspect
    # inspect.signature lists the names of a function's arguments, in order
    params = list(inspect.signature(bayes_ab).parameters)
    assert params[:4] == ["successes_a", "n_a", "successes_b", "n_b"] and "n_mc" in params \
        and "seed" in params, (
        f"bayes_ab's arguments are ({', '.join(params)}). The check calls it as the task "
        "asks: bayes_ab(successes_a, n_a, successes_b, n_b, n_mc=100_000, seed=42).")
    got = bayes_ab(357, 3000, 388, 3000)
    assert got is not None, "bayes_ab gives back nothing: use return, not print."
    assert isinstance(got, (tuple, list, np.ndarray)) and len(got) == 2, (
        f"bayes_ab should return two numbers, as return p, loss: P(B > A) and the expected "
        f"loss of shipping B. It returned {type(got).__name__} {got!r:.60}.")
    prob, loss = (float(v) for v in got)

    # Part 4's counts: these should reproduce Part 4's checkpoint, 0.8866 and 0.000473
    assert not (prob < 0.05 and loss > 0.5), (
        f"bayes_ab returned ({prob:.4f}, {loss:.4f}): the two numbers are in the wrong "
        "order. Return P(B > A) first, then the expected loss.")
    assert not (prob > 0.999 and loss < 1e-6), (
        f"On Part 4's counts P(B > A) came out {prob:.4f} with a loss of {loss:.6f}. A's and "
        "B's draws used the same seed, so they move together; use seed for A and seed + 1 for B.")
    assert abs(prob - 0.1134) > 0.005, (
        f"On Part 4's counts bayes_ab gives {prob:.4f}, which is 1 - 0.8866: that is "
        "P(A > B). Compare B's draws with A's, not the other way round.")
    assert abs(prob - 0.8567) > 0.005, (
        f"On Part 4's counts bayes_ab gives {prob:.4f}, not 0.8866. The second Beta parameter "
        "counts the users who did NOT convert, not all the users.")
    assert abs(prob - 0.8866) < 0.005, (
        f"On Part 4's counts bayes_ab gives P(B > A) = {prob:.4f}; Part 4's checkpoint is "
        "0.8866. Compare your posterior parameters with the formula at the top of Part 4: "
        "they are built from counts, not rates.")
    assert loss >= 0, (
        f"The expected loss came out negative ({loss:.6f}). A loss counts only the draws "
        "where A is better, so each draw adds zero or more.")
    assert abs(loss - 0.010770) > 0.0005, (
        f"The expected loss is {loss:.6f}: that is the loss of shipping A, from the draws "
        "where B is better. Shipping B loses in the draws where A is better.")
    assert abs(loss - 0.004168) > 0.0002, (
        f"The expected loss is {loss:.6f}: you averaged only the draws where A is better. "
        "Average over all draws, counting 0 where B is better.")
    assert abs(loss - 0.000473) < 0.00005, (
        f"On Part 4's counts the expected loss of shipping B is {loss:.6f}; Part 4's "
        "checkpoint is 0.000473.")

    # A small test, 2 of 10 against 5 of 10, where the prior matters
    small_prob, small_loss = (float(v) for v in bayes_ab(2, 10, 5, 10))
    assert abs(small_prob - prob) > 0.01, (
        "bayes_ab(2, 10, 5, 10) gives the same answer as Part 4's counts. Does the "
        "function use its arguments, or Part 4's successes_a and successes_b?")
    assert abs(small_prob - 0.9348) > 0.008, (
        f"bayes_ab(2, 10, 5, 10) gives P(B > A) = {small_prob:.4f}, the answer with no prior. "
        "With 10 users per arm the Beta(1, 1) prior matters: add it to both parameters.")
    assert abs(small_prob - 0.9087) < 0.005 and abs(small_loss - 0.0082) < 0.0005, (
        f"bayes_ab(2, 10, 5, 10) gives ({small_prob:.4f}, {small_loss:.6f}); it should be "
        "about (0.9087, 0.0082).")

    # The seed and n_mc arguments
    again = tuple(float(v) for v in bayes_ab(2, 10, 5, 10))
    assert again == (small_prob, small_loss), (
        "Two calls with the same seed gave different answers. Draw with "
        "random_state=seed and seed + 1, so each call can be repeated.")
    other_seed = tuple(float(v) for v in bayes_ab(2, 10, 5, 10, seed=7))
    assert other_seed != (small_prob, small_loss), (
        "seed=7 and seed=42 give identical answers: bayes_ab ignores its seed argument. "
        "Use seed, not the numbers 42 and 43.")
    few_prob, _ = bayes_ab(357, 3000, 388, 3000, n_mc=10)
    assert abs(few_prob * 10 - round(few_prob * 10)) < 1e-9, (
        f"With n_mc=10, P(B > A) should be a share of 10 draws; it is {few_prob:.5f}. "
        "Pass n_mc on as the number of draws.")

    # The longer test
    for name in ("longer_prob", "longer_loss", "longer_ship"):
        assert name in globals(), f"bayes_ab works. Now set {name}, as the task asks."
    long_prob, long_loss = (float(v) for v in bayes_ab(1428, 12000, 1560, 12000))
    assert (float(longer_loss), float(longer_prob)) != (long_prob, long_loss), (
        "longer_prob and longer_loss are the wrong way round: bayes_ab returns P(B > A) "
        "first, so unpack it as longer_prob, longer_loss = bayes_ab(...).")
    assert (float(longer_prob), float(longer_loss)) == (long_prob, long_loss), (
        f"longer_prob and longer_loss ({float(longer_prob):.4f}, {float(longer_loss):.6f}) are "
        "not bayes_ab on the longer test: 1,428 of 12,000 for A and 1,560 of 12,000 for B.")
    assert bool(longer_ship) is True, (
        f"With P(B > A) = {long_prob:.4f} and an expected loss of {long_loss:.6f}, apply "
        "Step E's rule to longer_prob and longer_loss: longer_ship is not what it gives.")
    print(f"bayes_ab matches Part 4: P(B > A) = {prob:.4f}, expected loss of shipping B {loss:.6f}.")
    print(f"Four times the users: P(B > A) = {long_prob:.4f}, expected loss {long_loss:.6f}, "
          f"ship B: {bool(longer_ship)}.")

---

## Module Output: Save the CUPED Utility (5 min)

Here is the finished module. `cuped_adjust` is Part 1's corrected CUPED as one
function, and it also accepts several covariates at once (it then regresses the
outcome on them and keeps the residual). `cuped_summary` prints a table of
specifications. Read `cuped_adjust` against your Part 1 fix, then run the three cells
below. They save it as `src/cuped_utils.py`, and the last one runs it on Part 1's
data: it should match your fixed numbers.

In [ ]:
# GIVEN — run as-is: make the src folder the next cell writes into
import os
os.makedirs("src", exist_ok=True)      # exist_ok: no error if it is already there

In [ ]:
%%writefile src/cuped_utils.py
"""CUPED variance reduction for A/B tests (ECON 5200, Lab 9).

After Deng et al. (2013), "Improving the Sensitivity of Online Controlled
Experiments by Utilizing Pre-Experiment Data."
"""

import numpy as np
from numpy.linalg import lstsq


def diff_in_means(y, treat_mask):
    """Treated mean minus control mean, and its standard error."""
    y_t, y_c = y[treat_mask], y[~treat_mask]        # ~ flips True and False
    ate = np.mean(y_t) - np.mean(y_c)
    se = np.sqrt(np.var(y_t, ddof=1) / len(y_t) + np.var(y_c, ddof=1) / len(y_c))
    return ate, se


def cuped_adjust(y, x_pre, treatment):
    """CUPED with one covariate (x_pre 1-D) or several (2-D). Returns a dict."""
    assert y.shape[0] == x_pre.shape[0] == treatment.shape[0]
    treat_mask = treatment == 1

    if x_pre.ndim == 1:
        # One covariate: theta = Cov(y, x) / Var(x), pooled across both arms
        cov_matrix = np.cov(y, x_pre)
        theta = cov_matrix[0, 1] / cov_matrix[1, 1]
        y_adj = y - theta * (x_pre - np.mean(x_pre))
    else:
        # Several covariates: regress y on them, keep the residual, add back y's mean
        X = np.column_stack([np.ones(len(y)), x_pre])
        # lstsq returns four things; _ names the three not used here
        betas, _, _, _ = lstsq(X, y, rcond=None)
        y_adj = y - X @ betas + np.mean(y)          # @ multiplies matrices: X @ betas is the fitted values

    ate_naive, se_naive = diff_in_means(y, treat_mask)
    ate_cuped, se_cuped = diff_in_means(y_adj, treat_mask)
    return {
        "y_adj": y_adj,
        "ate_naive": ate_naive,
        "se_naive": se_naive,
        "ate_cuped": ate_cuped,
        "se_cuped": se_cuped,
        "variance_reduction_pct": round((1 - (se_cuped / se_naive) ** 2) * 100, 2),
    }


def cuped_summary(y, x_pre_dict, treatment):
    """Print the naive estimate and one CUPED row per {label: covariate(s)}."""
    ate_naive, se_naive = diff_in_means(y, treatment == 1)
    # :<25 left-aligns in 25 characters; :>12,.2f right-aligns in 12, with commas and 2 decimals
    print(f"{'Method':<25} {'ATE':>12} {'SE':>12} {'Var Red':>10}")
    print("=" * 60)
    print(f"{'Naive':<25} {ate_naive:>12,.2f} {se_naive:>12,.2f} {'---':>10}")
    for label, x_pre in x_pre_dict.items():
        result = cuped_adjust(y, x_pre, treatment)
        print(f"{'CUPED (' + label + ')':<25} {result['ate_cuped']:>12,.2f} "
              f"{result['se_cuped']:>12,.2f} {result['variance_reduction_pct']:>9.1f}%")

In [ ]:
# GIVEN — run as-is: import the module and run it on Part 1's data
import sys
import importlib

sys.path.insert(0, "src")              # let Python find modules in src/
import cuped_utils
importlib.reload(cuped_utils)          # re-read the file, in case you ran the cell above again

summary = cuped_utils.cuped_adjust(y, x_pre, treatment)
print(f"CUPED ATE: {summary['ate_cuped']:.4f}  (SE = {summary['se_cuped']:.4f})")
print(f"Variance reduction: {summary['variance_reduction_pct']:.1f}%")

---

## Extension: Sequential Testing Simulation (10 min)

**The peeking problem:** In practice, experimenters rarely wait until the full sample is collected before checking results. They "peek" — computing p-values as data accumulates. Each p-value is valid on its own, but the chance that at least one of many looks falls below 0.05 is far above 5%.

This section simulates the peeking problem and demonstrates a simple correction.

In [ ]:
# GIVEN — run as-is
# Step 5: the peeking problem in one experiment

np.random.seed(123)

# One experiment in which H0 is true: both groups convert at 10%
n_max = 5000            # the planned sample size per group
true_rate = 0.10
data_a = np.random.binomial(1, true_rate, size=n_max)
data_b = np.random.binomial(1, true_rate, size=n_max)

# Peek every 100 users per group: a Welch t-test on the data collected so far
check_points = np.arange(100, n_max + 1, 100)       # 100, 200, ..., 5000
p_values_over_time = []
for n_current in check_points:
    _, p = stats.ttest_ind(data_b[:n_current], data_a[:n_current], equal_var=False)
    p_values_over_time.append(p)
p_values_over_time = np.array(p_values_over_time)

sig_mask = p_values_over_time < 0.05               # True at each peek with p < 0.05

In [ ]:
# GIVEN — run as-is
# Step 5, continued: the "dancing p-value"

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(check_points, p_values_over_time, color="steelblue")
ax.axhline(y=0.05, color="red", linestyle="--", label="alpha = 0.05")
# Indexing with sig_mask keeps only the peeks where p < 0.05
ax.scatter(check_points[sig_mask], p_values_over_time[sig_mask], color="red",
           label=f"p < 0.05 ({sig_mask.sum()} times)")
ax.set_xlabel("Sample Size per Group")
ax.set_ylabel("P-value")
ax.set_title("The Dancing P-Value: Peeking Under a True Null")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

print(f"P-value crossed below 0.05 a total of {sig_mask.sum()} times out of {len(check_points)} checks.")
# [0] picks the first of the peeks with p < 0.05
print(f"First crossing at n = {check_points[sig_mask][0]} per group, "
      f"p = {p_values_over_time[sig_mask][0]:.4f}.")
print(f"P-value at the planned end of the experiment (n = {check_points[-1]}): "
      f"{p_values_over_time[-1]:.4f}")

H0 is true here: both arms convert at exactly 10%. An experimenter who peeked and
stopped at the first p < 0.05 would have shipped a nonexistent effect at n = 700 per
group, and never seen the p-value climb back to 0.28 by the planned end.

### What the simulation does

The cell below repeats Step 5's experiment 1,000 times, each with no effect, and peeks
at every 100 users per group. It counts three things:

- **Step A:** the experiments in which the p-value falls below 0.05 at *any* of the 50 peeks.
- **Step B:** the same with a Bonferroni threshold, 0.05 / 50 = 0.001 at each peek.
- **Step C:** the experiments with p < 0.05 in a single test at the end, on all 5,000 users per group.

It reuses `n_max`, `true_rate` and `check_points` from Step 5, and takes about half a minute.

In [ ]:
# GIVEN — run as-is
# Steps A to C: 1,000 experiments with no effect, each peeked at every 100 users per group

np.random.seed(RANDOM_STATE)
n_sims = 1000
n_peeks = len(check_points)          # 50 peeks, as in Step 5
alpha = 0.05
alpha_bonferroni = alpha / n_peeks   # Step B: the threshold for each peek

fp_uncorrected = 0      # experiments with p < alpha at ANY peek
fp_bonferroni = 0       # experiments with p < alpha_bonferroni at any peek
fp_single_test = 0      # experiments with p < alpha in one test at the end

for sim in range(n_sims):
    a = np.random.binomial(1, true_rate, size=n_max)
    b = np.random.binomial(1, true_rate, size=n_max)

    ever_sig_uncorrected = False
    ever_sig_bonferroni = False
    for n_cur in check_points:
        _, p = stats.ttest_ind(b[:n_cur], a[:n_cur], equal_var=False)
        if p < alpha:
            ever_sig_uncorrected = True
        if p < alpha_bonferroni:                     # Step B
            ever_sig_bonferroni = True

    _, p_end = stats.ttest_ind(b, a, equal_var=False)    # Step C: one test, on all n_max users per group

    if ever_sig_uncorrected:
        fp_uncorrected += 1
    if ever_sig_bonferroni:
        fp_bonferroni += 1
    if p_end < alpha:
        fp_single_test += 1

print(f"=== False Positive Rates ({n_sims} simulations, true null) ===")
print(f"Uncorrected peeking ({n_peeks} peeks):  {fp_uncorrected / n_sims:.3f}")
print(f"Bonferroni-corrected (alpha/{n_peeks}): {fp_bonferroni / n_sims:.3f}")
print(f"Single test at end (no peeking):        {fp_single_test / n_sims:.3f}")
print(f"\nBonferroni alpha per peek: {alpha_bonferroni:.5f}")

**Verification checkpoint.** The measured false positive rates are **0.324** with
uncorrected peeking, **0.019** with Bonferroni at 0.05 / 50 = 0.001 per peek, and
**0.053** for a single test at the end. With 1,000 simulations each rate has a Monte
Carlo standard error of up to 1.5 points.

---

## Challenge (Take-Home): Full CUPED Pipeline on LaLonde

The challenge is optional and ungraded.

Apply CUPED to the LaLonde NSW data to reduce variance of the treatment effect estimate on `re78` (post-experiment earnings).

**Pre-experiment covariates:** Use `re74` (1974 earnings) and `re75` (1975 earnings) — both measured before the NSW program began.

**Your pipeline should:**
1. Compute the naive ATE and SE on `re78` (no adjustment)
2. Apply single-covariate CUPED using `re75` alone
3. Apply multi-covariate CUPED using both `re74` and `re75` (use OLS residualization)
4. Compare all three standard errors

Run the next cell to download the data first. Part 1's fixed code is the pattern. When you are done, `cuped_utils.cuped_summary` from the Module section prints the same table: use it to check your numbers, not to replace them.

In [ ]:
# GIVEN — run as-is, if you do the Challenge
# Load the LaLonde NSW dataset

# Dehejia & Wahba (2002) version of the LaLonde NSW experimental sample,
# served by NBER; .dta is a Stata file. 445 people: 185 treated, 260 control.
url = "https://users.nber.org/~rdehejia/data/nsw_dw.dta"
lalonde = pd.read_stata(url)

print(f"Shape: {lalonde.shape}")
print(f"\nTreatment split:")
print(lalonde["treat"].value_counts())          # how many rows have each value of treat
print(f"\nKey variables (first 5 rows):")
lalonde[["treat", "re74", "re75", "re78"]].head()

In [ ]:
# YOUR TASK — optional, ungraded: take-home Challenge, CUPED on the LaLonde data
# Write your code below this outline.

# Step 1: the naive ATE of treat on re78, and its SE

# Step 2: CUPED with re75 as the one covariate: theta, the ATE, its SE, the variance reduction

# Step 3: CUPED with re74 and re75 together: regress re78 on both (with a constant),
#         and use re78 minus the fitted values, plus the mean of re78, as the adjusted outcome

# Step 4: a table of the three ATEs, SEs and variance reductions

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Found and fixed three bugs in an analyst's CUPED (variance reduction)
  code. The analyst reported a [YOUR VALUE]% variance reduction; after my
  fixes it was [YOUR VALUE]%, with the SE falling from [YOUR VALUE] to
  [YOUR VALUE]
* Ran sample ratio mismatch tests on two experiments: a [YOUR VALUE] pp
  imbalance among 100,000 users gave chi-squared [YOUR VALUE], and a
  [YOUR VALUE] pp imbalance among 20 million users gave [YOUR VALUE]
* Simulated a marketplace where discounted buyers take inventory from the
  control group: the A/B test's naive ATE was [YOUR VALUE], against
  [YOUR VALUE] for rolling the discount out to everyone, an overstatement
  of [YOUR VALUE]% of the naive estimate
* Ran a Bayesian A/B test: P(B > A) = [YOUR VALUE] and the expected loss of
  shipping B was [YOUR VALUE], so the ship rule said [YOUR VALUE]; a
  permutation test gave p = [YOUR VALUE]
* Wrote my own function for the Bayesian test and used it on a test with
  four times the users: P(B > A) = [YOUR VALUE]
* Simulated peeking at 50 points in 1,000 experiments with no effect:
  [YOUR VALUE]% crossed p < 0.05 at least once, against [YOUR VALUE]% with
  a single test at the end
* (If I did the challenge) Applied CUPED to the LaLonde job training data:
  re74 and re75 cut the variance by [YOUR VALUE]%

**Please write a README.md entry including:**
1. Project Title: Diagnosing Experimental Failures in A/B Testing
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab09-ab-testing`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab09-ab-testing`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab09-ab-testing`, branch `main`,
   commit message `Lab 09 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/cuped_utils.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.